# Vinos: limpieza, normalización y comparación
Ejecuta las celdas en orden. No necesitas montar Google Drive. COVID se añadirá después de revisar su archivo.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL = "https://github.com/TU_USUARIO/ml-vinos-covid.git"
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
if EN_COLAB:
    if "TU_USUARIO" in REPO_URL:
        raise ValueError("Reemplaza REPO_URL por la URL de tu repositorio en GitHub.")
    destino = Path("/content/ml-vinos-covid")
    if not destino.exists():
        subprocess.run(["git", "clone", REPO_URL, str(destino)], check=True)
    os.chdir(destino)
else:
    inicio = Path.cwd().resolve()
    raiz = next((p for p in [inicio, *inicio.parents] if (p / "src/vinos.py").exists()), None)
    if raiz is None:
        raise FileNotFoundError("Abre el notebook dentro del proyecto descomprimido o clonado.")
    os.chdir(raiz)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
# Si Colab pide reiniciar la sesión después de instalar, reinicia y vuelve a ejecutar.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print("Proyecto:", Path.cwd())

## 1. Revisar y limpiar
Se conserva el original. Eliminamos duplicados exactos y comprobamos tipos/nulos; no modificamos quality.

In [ ]:
from src.vinos import load_clean, run
from IPython.display import display
clean, reporte = load_clean()
print(reporte)
display(clean.head())
display(clean.describe())
display(clean.quality.value_counts().sort_index())

## 2. Separar, normalizar y entrenar
El código compartido en src/vinos.py separa 80/20; usa imputación y MinMax dentro de Pipeline. La CV usa solo entrenamiento. Se comparan lineal/Ridge, árbol/Random Forest y SVR lineal/RBF. El recorte IQR está desactivado por defecto; los atípicos no siempre son errores.

In [ ]:
comparacion_cv, comparacion_test, reporte = run(cap_outliers=False)

## 3. Interpretar
Elige por MAE_CV (menor es mejor); usa test únicamente como evaluación final. MAE y RMSE están en puntos de calidad; R² no es un porcentaje de aciertos. Compara con la referencia de la media. No reajustes los modelos mirando test.

In [ ]:
display(comparacion_cv.round(4))
display(comparacion_test.round(4))

## 4. Preguntas para el informe
- ¿Qué se detectó y qué se modificó en limpieza?
- ¿Por qué normalizamos después de separar?
- ¿Qué modelo tiene menor MAE de CV en cada área?
- ¿Mejora la referencia? ¿Cuánto varía entre folds?
- ¿Qué limitaciones tienen la puntuación ordinal y esta partición?

Los resultados están en results/vinos/. Se regeneran en cada ejecución. Para actualizar código en una sesión Colab existente, usa una sesión nueva o actualiza el clon conscientemente; la primera celda no sobrescribe tus cambios.